# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue:", total_revenue)
print("Total units:", total_units)

Total revenue: 8520.0
Total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [4]:
# TODO
by_category = df.groupby('category').agg(
    revenue=('revenue', 'sum')
)

by_category = by_category.sort_values(
    'revenue',
    ascending=False
)

by_category['share_pct'] = (
    by_category['revenue'] / total_revenue * 100
).round(1)

by_category

,revenue,share_pct
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [5]:
# TODO
by_vendor = df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).round(2)

by_vendor = by_vendor.sort_values(
    'avg_order_revenue',
    ascending=False
)

by_vendor

,avg_order_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [11]:
# TODO
merch = df[df['category'] == 'Merch']

merch_revenue = merch['revenue'].sum()

merch_share = merch_revenue / total_revenue * 100
merch_share = round(merch_share, 1)

print(merch_share, "%")

20.8 %


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [12]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    indicator=True,
    validate='many_to_one'
)

print('rows before:', len(df), '| rows after:', len(joined))

print('revenue before:', df['revenue'].sum())
print('revenue after:', joined['revenue'].sum())

unmatched = joined[joined['_merge'] == 'left_only']

print('unmatched vendor:', unmatched['vendor_id'].unique())

rows before: 400 | rows after: 400
revenue before: 8520.0
revenue after: 8520.0
unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** V-18 was the unmatched vendor. I kept its orders in the dataset because the left join preserves all original orders, but its vendor name remains missing because it was not provided in the lookup table

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [13]:
# TODO
pivot = pd.pivot_table(
    df,
    index='vendor_id',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [14]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.


a)For the next game, I would tell the vendors to focus more on food which generated around 50.4% of total revenue and maybe less on RainGear which was only about 10.6%. So vendors may want to carry less/supply less RainGear and instead focus on Food and Merch, which generated the most revenue. I also wanted to point out that, although V-18 generated the most total revenue, V-01 had the most amount of revenue per order, at around $22.60 per order. So maybe look into what vendor v-01 did to see how to get customers to spend the most amount of money.


b)The least trustworthy answer are the ones that included information on vendor V-18, so maybe question 5. Because V-18 was not included in the look up table but it is still included because we used a left join. We cannot confirm the actual vendor name from the data provided.

